# Parcel Address Associator: end-to-end session

Finds **potential duplicate addresses within each cadastre parcel** by working out the address *styles* present (the schematics) and learning how those styles map onto each other. It's built for complex apartment precincts, where one dwelling is often addressed several ways:

| style | example |
|---|---|
| building prefix + level-encoded number, range base | `UNIT A1103, 1-15 MERIDIAN AV` |
| building name | `TOWER A, UNIT 1103, 1-15 MERIDIAN AV` |
| explicit level, unit restarts per level | `UNIT 3, LEVEL 11, 1 MERIDIAN AV` |

Matching is rule-based and statistical only: no LLMs and no libpostal.

**Pipeline** (each stage is a cell below):

```
input DataFrame ─► 1 validate ─► 2 profile ─► 3 run (per parcel) ─► 4 overview
                                                 │
   per parcel: features ─► styles ─► blocks + numbering models ─► block mappings (transform + evidence)
               ─► inferred structures (stacking plans) ─► duplicate pairs ─► duplicate groups + review list
```

**Input:** one row per address, already linked to a parcel. Columns: `ADDR_ID` (identifier), `SOURCE_ID` (address text identifier), the G-NAF component columns, and `CADASTRE_POLYGON_PID`. Only `ADDR_ID` and `CADASTRE_POLYGON_PID` are required; missing component columns are treated as blank.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))          # project root (this notebook lives in notebooks/)

import pandas as pd
from parcel_address_associator import AssociatorSession, Config, INPUT_COLUMNS, make_synthetic

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 220)
print(len(INPUT_COLUMNS), "input columns:", ", ".join(INPUT_COLUMNS))

27 input columns: ADDR_ID, SOURCE_ID, BUILDING_NAME, LOT_NUMBER_PREFIX, LOT_NUMBER, LOT_NUMBER_SUFFIX, FLAT_TYPE_CODE, FLAT_NUMBER_PREFIX, FLAT_NUMBER, FLAT_NUMBER_SUFFIX, LEVEL_TYPE_CODE, LEVEL_NUMBER_PREFIX, LEVEL_NUMBER, LEVEL_NUMBER_SUFFIX, NUMBER_FIRST_PREFIX, NUMBER_FIRST, NUMBER_FIRST_SUFFIX, NUMBER_LAST_PREFIX, NUMBER_LAST, NUMBER_LAST_SUFFIX, STREET_NAME, STREET_TYPE_CODE, STREET_SUFFIX_CODE, LOCALITY_NAME, STATE_ABBREVIATION, POSTCODE, CADASTRE_POLYGON_PID


## 1 · Input

Swap the synthetic data for your own DataFrame, e.g. `df = spark.table("...").toPandas()` or `df = pd.read_parquet(...)`.

The synthetic set is a fictional town of apartment precincts. Each one reproduces a duplicate pattern from real data, rendered by several "sources" that number the same dwellings differently. `truth` records which physical dwelling each `ADDR_ID` is, and is only used for scoring in section 9.

In [2]:
df, truth = make_synthetic(seed=42)
# df = <your DataFrame with INPUT_COLUMNS>
print(df.shape)
df.head(8)

(2100, 27)


,ADDR_ID,SOURCE_ID,BUILDING_NAME,LOT_NUMBER_PREFIX,LOT_NUMBER,LOT_NUMBER_SUFFIX,FLAT_TYPE_CODE,FLAT_NUMBER_PREFIX,FLAT_NUMBER,FLAT_NUMBER_SUFFIX,LEVEL_TYPE_CODE,LEVEL_NUMBER_PREFIX,LEVEL_NUMBER,LEVEL_NUMBER_SUFFIX,NUMBER_FIRST_PREFIX,NUMBER_FIRST,NUMBER_FIRST_SUFFIX,NUMBER_LAST_PREFIX,NUMBER_LAST,NUMBER_LAST_SUFFIX,STREET_NAME,STREET_TYPE_CODE,STREET_SUFFIX_CODE,LOCALITY_NAME,STATE_ABBREVIATION,POSTCODE,CADASTRE_POLYGON_PID
0,A0000943,TXT40012259,TOWER C,,,,UNIT,,503,,,,,,,1,,,15,,MERIDIAN,AVENUE,,PORT ALDER,NSW,2989,cpl5a1700003dde
1,A0001407,TXT40018291,,,,,UNIT,A,804,,,,,,,50,,,52,,SKYLINE,BOULEVARD,,PORT ALDER,NSW,2989,cpl5a170000b99a
2,A0000296,TXT40003848,,,,,UNIT,A,409,,,,,,,1,,,15,,MERIDIAN,AVENUE,,PORT ALDER,NSW,2989,cpl5a1700003dde
3,A0001211,TXT40015743,,,,,UNIT,,806,,,,,,,120,,,,,CANAL,ROAD,,PORT ALDER,NSW,2989,cpl5a1700007bbc
4,A0001425,TXT40018525,TOWER A,,,,UNIT,,1002,,,,,,,50,,,52,,SKYLINE,BOULEVARD,,PORT ALDER,NSW,2989,cpl5a170000b99a
5,A0001026,TXT40013338,,,,,UNIT,,304,,,,,,,40,,,,,QUARRY,STREET,,PORT ALDER,NSW,2989,cpl5a1700005ccd
6,A0001803,TXT40023439,,,,,SHOP,,2,,,,,,,64,,,68,,BANKSIA,STREET,,PORT ALDER,NSW,2989,cpl5a1700011667
7,A0001084,TXT40014092,,,,,UNIT,,806,,,,,,,2,,,,,WATTLE,LANE,,PORT ALDER,NSW,2989,cpl5a1700005ccd


## 2 · Validate

`validate()` checks the schema and repairs what it safely can. Problems are reported, not fatal:
- numbers stored as floats (`101.0` → `101`);
- identifiers packed into the number field (`G01` → prefix `G` + number `1`);
- names instead of codes (`APARTMENT` → `APT`, `ST` → `STREET`).

It also flags blank or duplicate `ADDR_ID`s, blank parcels and unknown codes.

In [3]:
s = AssociatorSession(df)
s.validate()

validate: 2100 of 2100 rows accepted, 1 issue types (0 error rows)


,CHECK,SEVERITY,ROWS,DETAIL,EXAMPLE_ADDR_IDS
0,rows_accepted,info,2100,2100 of 2100 input rows accepted,


What the checks look like on messy input (a few rows deliberately broken):

In [4]:
messy = df.head(6).astype(object)
messy["LEVEL_NUMBER"] = pd.to_numeric(messy["LEVEL_NUMBER"], errors="coerce")  # float column with NaN
messy.loc[0, "FLAT_NUMBER"] = 101.0                      # number stored as float
messy.loc[1, ["FLAT_NUMBER_PREFIX", "FLAT_NUMBER"]] = ["", "G01"]   # packed identifier
messy.loc[2, "FLAT_TYPE_CODE"] = "APARTMENT"             # name instead of code
messy.loc[3, "STREET_TYPE_CODE"] = "ST"                  # abbreviation instead of code
messy.loc[4, "ADDR_ID"] = messy.loc[5, "ADDR_ID"]        # duplicate id
messy = messy.drop(columns=["SOURCE_ID"])                # missing optional column
AssociatorSession(messy, verbose=False).validate()

,CHECK,SEVERITY,ROWS,DETAIL,EXAMPLE_ADDR_IDS
0,missing_column,warning,6,SOURCE_ID not supplied; treated as blank,
1,duplicate_addr_id,error,1,repeated ADDR_ID; first occurrence kept,A0001026
2,code_normalised,info,1,FLAT_TYPE_CODE names/abbreviations mapped to G-NAF codes (e.g. APARTMENT->APT),
3,code_normalised,info,1,STREET_TYPE_CODE names/abbreviations mapped to G-NAF codes (e.g. ST->STREET),
4,identifier_split,info,1,FLAT_NUMBER held prefix/suffix letters (e.g. G01); split into FLAT_NUMBER_PR...,A0001407
5,rows_accepted,info,5,5 of 6 input rows accepted,


## 3 · Profile

What each parcel holds before any matching. Parcels with fewer than `Config.min_addresses` addresses (default 2) are skipped, because there is nothing to deduplicate.

In [5]:
s.profile(top=10)

profile: 49 parcels, 28 with >= 2 addresses will be analysed


,CADASTRE_POLYGON_PID,N_ADDRESSES,N_SUB_ADDRESSES,N_WITH_LEVEL,N_WITH_BUILDING_NAME,N_RANGE_ADDRESSES,N_DISTINCT_BASES,FLAT_TYPES,WILL_ANALYSE
0,cpl5a1700003dde,790,783,0,261,593,4,"UNIT:773, SHOP:10",True
1,cpl5a170000b99a,219,216,0,98,219,1,UNIT:216,True
2,cpl5a170000d889,211,206,0,88,89,3,UNIT:206,True
3,cpl5a1700001eef,201,198,66,1,134,2,"UNIT:178, APT:13, SHOP:7",True
4,cpl5a1700011667,163,161,3,0,105,2,"UNIT:88, APT:46, SE:10, OFFC:8, SHOP:6",True
5,cpl5a1700007bbc,118,117,0,0,0,1,UNIT:117,True
6,cpl5a1700005ccd,117,115,24,0,0,2,UNIT:115,True
7,cpl5a1700009aab,91,90,36,0,0,1,UNIT:90,True
8,cpl5a170000f778,51,49,0,0,0,2,"UNIT:30, FLAT:11, TNHS:8",True
9,cpl5a170005ccd0,10,9,0,0,0,1,"UNIT:5, FLAT:4",True


## 4 · Run

Every parcel is analysed independently: features → styles → blocks → block mappings → structures → pairs → groups. `s.log` records each stage.

In [6]:
s.run()
s.log

run: 28 parcels analysed in 0.30s, 655 duplicate groups


,STAGE,SECONDS,DETAIL
0,validate,0.156,"rows_in=2100, rows_accepted=2100, issues=1, error_rows=0"
1,profile,0.018,"parcels=49, parcels_to_analyse=28"
2,run,0.300,"parcels=28, groups=655, review_pairs=226"


## 5 · Overview

In [7]:
s.overview()
s.parcel_summary().head(10)

,CADASTRE_POLYGON_PID,N_ADDRESSES,N_SUB_ADDRESSES,N_PARENTS,N_STYLES,N_BLOCKS,N_STRUCTURES,N_DUPLICATE_GROUPS,N_ADDRESSES_IN_GROUPS,N_REDUNDANT_ADDRESSES,N_REVIEW_PAIRS,REJECTED_CONFLICTS
0,cpl5a1700003dde,790,783,7,5,12,3,305,760,455,0,0
1,cpl5a170000b99a,219,216,3,2,4,2,96,192,96,0,0
2,cpl5a1700001eef,201,198,3,5,5,1,75,159,84,0,9
3,cpl5a1700011667,163,161,2,6,6,1,58,116,58,0,0
4,cpl5a1700005ccd,117,115,2,2,3,1,44,96,52,0,0
5,cpl5a1700009aab,91,90,1,2,2,1,36,72,36,0,0
6,cpl5a170000f778,51,49,2,3,4,2,17,34,17,0,0
7,cpl5a170005ccd0,10,9,1,2,2,1,4,8,4,0,0
8,cpl5a1700057003,10,9,1,2,2,1,3,6,3,0,0
9,cpl5a1700017334,9,8,1,2,2,1,3,6,3,0,0


## 6 · One parcel, stage by stage

Taking the parcel with the most duplicate groups. Vocabulary used below:

- **Style:** a formatting scheme, e.g. `UNIT {B}{L}{UU}, {N}-{N} {STREET}` (building prefix + level-encoded unit on a range base).
- **Block:** the addresses of one style on one context (base address + building reference). It is what that scheme says is in one building.
- **Numbering model:** how a block's numbers encode position: `encoded` (1103 = level 11 unit 3), `explicit+per-level` (`UNIT 3, LEVEL 11`), `explicit+sequential`, `prefix-level` (G01), `plain`.
- **Mapping:** for each pair of blocks, the transform that explains the most records (same identifier; level + unit-on-level, optionally with a level offset; position within level; position through the building). It is scored in log-odds from coverage, layout extent, building references and base-address relationship.
- **Structure:** linked blocks clustered into an inferred building, drawn as a stacking plan.

In [8]:
pid = s.interesting_parcels(1)[0]
print(s.schematic(pid))

PARCEL cpl5a1700003dde  790 addresses (783 sub-addresses, 7 parents) | 305 duplicate groups | 0 review
  STYLES
    S1   UNIT {B}{L}{UU}, {N}-{N} {STREET}                         324  UNIT|BPFX|encoded|RANGE
         S1.1    1-15 MERIDIAN AV | prefix A            n=180  unit number encodes level (1103 = level 11, unit 3); levels L1-L18, ~10 per level
         S1.2    1-15 MERIDIAN AV | prefix B            n=96   unit number encodes level (1103 = level 11, unit 3); levels L1-L12, ~8 per level
         S1.3    1-15 MERIDIAN AV | prefix C            n=48   unit number encodes level (1103 = level 11, unit 3); levels L1-L8, ~6 per level
    S2   TOWER {B}, UNIT {L}{UU}, {N}-{N} {STREET}                 258  UNIT|encoded|BN|RANGE
         S2.1    1-15 MERIDIAN AV | TOWER A             n=144  unit number encodes level (1103 = level 11, unit 3); levels L1-L18, ~8 per level
         S2.2    1-15 MERIDIAN AV | TOWER B             n=79   unit number encodes level (1103 = level 11, unit 3); levels

### 6a · Styles on the parcel

In [9]:
s.styles(pid)

,CADASTRE_POLYGON_PID,STYLE_ID,TEMPLATE,SIGNATURE,N_ADDRESSES,N_BLOCKS,BLOCKS,EXAMPLES
0,cpl5a1700003dde,S1,"UNIT {B}{L}{UU}, {N}-{N} {STREET}",UNIT|BPFX|encoded|RANGE,324,3,S1.1: 1-15 MERIDIAN AV | prefix A [encoded] n=180; S1.2: 1-15 MERIDIAN AV | ...,"UNIT A409, 1-15 MERIDIAN AV | UNIT B1002, 1-15 MERIDIAN AV | UNIT A1509, 1-1..."
1,cpl5a1700003dde,S2,"TOWER {B}, UNIT {L}{UU}, {N}-{N} {STREET}",UNIT|encoded|BN|RANGE,258,3,S2.1: 1-15 MERIDIAN AV | TOWER A [encoded] n=144; S2.2: 1-15 MERIDIAN AV | T...,"TOWER C, UNIT 503, 1-15 MERIDIAN AV | TOWER A, UNIT 1006, 1-15 MERIDIAN AV |..."
2,cpl5a1700003dde,S3,"UNIT {L}{UU}, {N} {STREET}",UNIT|encoded|SINGLE,191,3,S3.1: 1 MERIDIAN AV [encoded] n=102; S3.2: 15 MERIDIAN AV [encoded] n=28; S3...,"UNIT 401, 15 MERIDIAN AV | UNIT 1404, 1 MERIDIAN AV | UNIT 203, 7 MERIDIAN AV"
3,cpl5a1700003dde,S4,"SHOP {N}, {N}-{N} {STREET}",SHOP|plain|RANGE,7,1,S4.1: 1-15 MERIDIAN AV [plain] n=7,"SHOP 7, 1-15 MERIDIAN AV | SHOP 6, 1-15 MERIDIAN AV | SHOP 4, 1-15 MERIDIAN AV"
4,cpl5a1700003dde,S5,"SHOP {N}, {N} {STREET}",SHOP|plain|SINGLE,3,2,S5.1: 1 MERIDIAN AV [plain] n=2; S5.2: 7 MERIDIAN AV [plain] n=1,"SHOP 2, 1 MERIDIAN AV | SHOP 3, 1 MERIDIAN AV | SHOP 6, 7 MERIDIAN AV"


### 6b · Blocks and numbering models

In [10]:
s.blocks(pid)

,CADASTRE_POLYGON_PID,BLOCK_ID,STYLE_ID,CONTEXT,BUILDING_REF,FLAT_TYPE,NUMBERING_MODEL,MODEL_NOTE,N_ADDRESSES,LEVELS,UNITS_PER_LEVEL,STRUCTURE_ID,SPANS_BUILDINGS,KEY_FORMS
0,cpl5a1700003dde,S1.1,S1,1-15 MERIDIAN AV | prefix A,A,UNIT,encoded,"unit number encodes level (1103 = level 11, unit 3); levels L1-L18, ~10 per ...",180,L1-L18,10.0,B1,False,"raw,lu,lr,seq"
1,cpl5a1700003dde,S1.2,S1,1-15 MERIDIAN AV | prefix B,B,UNIT,encoded,"unit number encodes level (1103 = level 11, unit 3); levels L1-L12, ~8 per l...",96,L1-L12,8.0,B2,False,"raw,lu,lr,seq"
2,cpl5a1700003dde,S1.3,S1,1-15 MERIDIAN AV | prefix C,C,UNIT,encoded,"unit number encodes level (1103 = level 11, unit 3); levels L1-L8, ~6 per level",48,L1-L8,6.0,B3,False,"raw,lu,lr,seq"
3,cpl5a1700003dde,S2.1,S2,1-15 MERIDIAN AV | TOWER A,A,UNIT,encoded,"unit number encodes level (1103 = level 11, unit 3); levels L1-L18, ~8 per l...",144,L1-L18,8.0,B1,False,"raw,lu,lr,seq"
4,cpl5a1700003dde,S2.2,S2,1-15 MERIDIAN AV | TOWER B,B,UNIT,encoded,"unit number encodes level (1103 = level 11, unit 3); levels L1-L12, ~7 per l...",79,L1-L12,7.0,B2,False,"raw,lu,lr,seq"
5,cpl5a1700003dde,S2.3,S2,1-15 MERIDIAN AV | TOWER C,C,UNIT,encoded,"unit number encodes level (1103 = level 11, unit 3); levels L1-L8, ~5 per level",35,L1-L8,5.0,B3,False,"raw,lu,lr,seq"
6,cpl5a1700003dde,S3.1,S3,1 MERIDIAN AV,,UNIT,encoded,"unit number encodes level (1103 = level 11, unit 3); levels L1-L18, ~6 per l...",102,L1-L18,6.0,B1,False,"raw,lu,lr,seq"
7,cpl5a1700003dde,S3.2,S3,15 MERIDIAN AV,,UNIT,encoded,"unit number encodes level (1103 = level 11, unit 3); levels L1-L8, ~3.5 per ...",28,L1-L8,3.5,B3,False,"raw,lu,lr,seq"
8,cpl5a1700003dde,S3.3,S3,7 MERIDIAN AV,,UNIT,encoded,"unit number encodes level (1103 = level 11, unit 3); levels L1-L12, ~5.5 per...",61,L1-L12,5.5,B2,False,"raw,lu,lr,seq"
9,cpl5a1700003dde,S4.1,S4,1-15 MERIDIAN AV,,SHOP,plain,"plain sequence, no level information",7,,NaN,,True,"raw,seq,num"


### 6c · How the blocks map to each other

`STATUS`:
- `KEPT`: used for duplicates.
- `REJECTED`: the evidence says these are different buildings or premises.
- `AMBIGUOUS`: fits two buildings equally well.
- `REVIEW`: an order-based mapping that components alone can't confirm.

`EVIDENCE` lists each log-odds contribution.

In [11]:
s.mappings(pid, include_rejected=True)

,CADASTRE_POLYGON_PID,LINK_ID,BLOCK_A,CONTEXT_A,BLOCK_B,CONTEXT_B,TRANSFORM,MATCHED_KEYS,KEYS_A,KEYS_B,QUALITY,P,BAND,STATUS,EVIDENCE
0,cpl5a1700003dde,S1.1~S2.1,S1.1,1-15 MERIDIAN AV | prefix A,S2.1,1-15 MERIDIAN AV | TOWER A,same unit identifier,144,180,144,1.000,0.9980,HIGH,KEPT,same unit identifier explains 144 of 180/144 keys (+3.2); building ref A = A...
1,cpl5a1700003dde,S1.2~S2.2,S1.2,1-15 MERIDIAN AV | prefix B,S2.2,1-15 MERIDIAN AV | TOWER B,same unit identifier,79,96,79,1.000,0.9980,HIGH,KEPT,same unit identifier explains 79 of 96/79 keys (+3.2); building ref B = B (+...
2,cpl5a1700003dde,S1.3~S2.3,S1.3,1-15 MERIDIAN AV | prefix C,S2.3,1-15 MERIDIAN AV | TOWER C,same unit identifier,35,48,35,1.000,0.9980,HIGH,KEPT,same unit identifier explains 35 of 48/35 keys (+3.2); building ref C = C (+...
3,cpl5a1700003dde,S1.1~S3.1,S1.1,1-15 MERIDIAN AV | prefix A,S3.1,1 MERIDIAN AV,same unit identifier,102,180,102,1.000,0.9820,HIGH,KEPT,same unit identifier explains 102 of 180/102 keys (+3.2); range contains num...
4,cpl5a1700003dde,S1.2~S3.3,S1.2,1-15 MERIDIAN AV | prefix B,S3.3,7 MERIDIAN AV,same unit identifier,61,96,61,1.000,0.9820,HIGH,KEPT,same unit identifier explains 61 of 96/61 keys (+3.2); range contains number...
5,cpl5a1700003dde,S1.3~S3.2,S1.3,1-15 MERIDIAN AV | prefix C,S3.2,15 MERIDIAN AV,same unit identifier,28,48,28,1.000,0.9820,HIGH,KEPT,same unit identifier explains 28 of 48/28 keys (+3.2); range contains number...
6,cpl5a1700003dde,S2.1~S3.1,S2.1,1-15 MERIDIAN AV | TOWER A,S3.1,1 MERIDIAN AV,same unit identifier,83,144,102,0.888,0.9571,HIGH,KEPT,same unit identifier explains 83 of 144/102 keys (+2.3); range contains numb...
7,cpl5a1700003dde,S2.2~S3.3,S2.2,1-15 MERIDIAN AV | TOWER B,S3.3,7 MERIDIAN AV,same unit identifier,47,79,61,0.862,0.9478,HIGH,KEPT,same unit identifier explains 47 of 79/61 keys (+2.1); range contains number...
8,cpl5a1700003dde,S2.3~S3.2,S2.3,1-15 MERIDIAN AV | TOWER C,S3.2,15 MERIDIAN AV,same unit identifier,20,35,28,0.829,0.9327,HIGH,KEPT,same unit identifier explains 20 of 35/28 keys (+1.8); range contains number...
9,cpl5a1700003dde,S4.1~S5.1,S4.1,1-15 MERIDIAN AV,S5.1,1 MERIDIAN AV,same unit identifier,2,7,2,0.714,0.7711,MEDIUM,KEPT,same unit identifier explains 2 of 7/2 keys (+0.9); only 2 matched (-0.5); r...


### 6d · Inferred structures

In [12]:
s.structures(pid)

,CADASTRE_POLYGON_PID,STRUCTURE_ID,LABEL,BLOCKS,CLASSES,LEVELS,N_SLOTS,N_SLOTS_WITH_DUPLICATES,N_UNPLACED
0,cpl5a1700003dde,B1,A | 1 MERIDIAN AV,"S1.1,S2.1,S3.1,S5.1","COMMERCIAL,RESIDENTIAL",L1-L18,180,163,2
1,cpl5a1700003dde,B2,B | 7 MERIDIAN AV,"S1.2,S2.2,S3.3,S5.2","COMMERCIAL,RESIDENTIAL",L1-L12,96,93,1
2,cpl5a1700003dde,B3,C | 15 MERIDIAN AV,"S1.3,S2.3,S3.2",RESIDENTIAL,L1-L8,48,43,0


### 6e · The full schematic

Styles, mappings, a stacking plan per inferred building, parents and groups. In each stacking-plan cell, the coloured stripes show which styles address that slot (the stripe position is fixed per style) and the number is how many addresses fall in it. Hover a cell to list them.

In [13]:
s.show_parcel(pid)

A parcel the tool *can't* resolve: two identical towers where one source uses street numbers and the other uses EAST/WEST TOWER names. Nothing in the components says which is which, so the pairs go to the review list instead of being guessed.

In [14]:
amb = s.parcel_summary().sort_values("N_REVIEW_PAIRS", ascending=False).CADASTRE_POLYGON_PID.iloc[0]
s.show_parcel(amb)

## 7 · Results

- `duplicate_groups()`: one row per group, with a suggested record to keep. The suggestion comes from the parcel's dominant style, then the most complete record.
- `duplicate_pairs()`: the evidence behind each grouped pair.
- `review_candidates()`: plausible pairs that components can't confirm. These are never grouped.
- `addresses()`: every analysed input row annotated with style, block, structure, slot and group.

In [15]:
groups = s.duplicate_groups()
print(groups.BAND.value_counts().to_dict(), "|", groups.RELATION.value_counts().to_dict())
groups.head(10)

{'HIGH': 611, 'LOW': 28, 'MEDIUM': 16} | {'SAME_DWELLING': 646, 'SAME_BUILDING': 6, 'SAME_PROPERTY': 2, 'EXACT_DUPLICATE': 1}


,CADASTRE_POLYGON_PID,GROUP_ID,SIZE,RELATION,BAND,P_MIN,P_MAX,PREFERRED_ADDR_ID,PREFERRED_LABEL,MEMBER_ADDR_IDS,MEMBER_SOURCE_IDS,MEMBER_LABELS,MEMBER_STYLES
0,cpl5a1700003dde,G1,2,SAME_BUILDING,MEDIUM,0.8455,0.8455,A0000204,1 MERIDIAN AV,A0000203;A0000204,TXT40002639;TXT40002652,"TOWER A, 1-15 MERIDIAN AV | 1 MERIDIAN AV",parent;parent
1,cpl5a1700003dde,G2,3,SAME_DWELLING,HIGH,0.9571,0.9980,A0000205,"UNIT A101, 1-15 MERIDIAN AV",A0000205;A0000206;A0000207,TXT40002665;TXT40002678;TXT40002691,"UNIT A101, 1-15 MERIDIAN AV | TOWER A, UNIT 101, 1-15 MERIDIAN AV | UNIT 101...",S1;S2;S3
2,cpl5a1700003dde,G3,2,SAME_DWELLING,HIGH,0.9980,0.9980,A0000208,"UNIT A102, 1-15 MERIDIAN AV",A0000208;A0000209,TXT40002704;TXT40002717,"UNIT A102, 1-15 MERIDIAN AV | TOWER A, UNIT 102, 1-15 MERIDIAN AV",S1;S2
3,cpl5a1700003dde,G4,3,SAME_DWELLING,HIGH,0.9571,0.9980,A0000210,"UNIT A103, 1-15 MERIDIAN AV",A0000210;A0000211;A0000212,TXT40002730;TXT40002743;TXT40002756,"UNIT A103, 1-15 MERIDIAN AV | TOWER A, UNIT 103, 1-15 MERIDIAN AV | UNIT 103...",S1;S2;S3
4,cpl5a1700003dde,G5,2,SAME_DWELLING,HIGH,0.9820,0.9820,A0000213,"UNIT A104, 1-15 MERIDIAN AV",A0000213;A0000214,TXT40002769;TXT40002782,"UNIT A104, 1-15 MERIDIAN AV | UNIT 104, 1 MERIDIAN AV",S1;S3
5,cpl5a1700003dde,G6,2,SAME_DWELLING,HIGH,0.9980,0.9980,A0000215,"UNIT A105, 1-15 MERIDIAN AV",A0000215;A0000216,TXT40002795;TXT40002808,"UNIT A105, 1-15 MERIDIAN AV | TOWER A, UNIT 105, 1-15 MERIDIAN AV",S1;S2
6,cpl5a1700003dde,G7,3,SAME_DWELLING,HIGH,0.9571,0.9980,A0000217,"UNIT A106, 1-15 MERIDIAN AV",A0000217;A0000218;A0000219,TXT40002821;TXT40002834;TXT40002847,"UNIT A106, 1-15 MERIDIAN AV | TOWER A, UNIT 106, 1-15 MERIDIAN AV | UNIT 106...",S1;S2;S3
7,cpl5a1700003dde,G8,2,SAME_DWELLING,HIGH,0.9980,0.9980,A0000220,"UNIT A107, 1-15 MERIDIAN AV",A0000220;A0000221,TXT40002860;TXT40002873,"UNIT A107, 1-15 MERIDIAN AV | TOWER A, UNIT 107, 1-15 MERIDIAN AV",S1;S2
8,cpl5a1700003dde,G9,2,SAME_DWELLING,HIGH,0.9980,0.9980,A0000222,"UNIT A108, 1-15 MERIDIAN AV",A0000222;A0000223,TXT40002886;TXT40002899,"UNIT A108, 1-15 MERIDIAN AV | TOWER A, UNIT 108, 1-15 MERIDIAN AV",S1;S2
9,cpl5a1700003dde,G10,3,SAME_DWELLING,HIGH,0.9571,0.9980,A0000224,"UNIT A109, 1-15 MERIDIAN AV",A0000224;A0000225;A0000226,TXT40002912;TXT40002925;TXT40002938,"UNIT A109, 1-15 MERIDIAN AV | TOWER A, UNIT 109, 1-15 MERIDIAN AV | UNIT 109...",S1;S2;S3


In [16]:
s.duplicate_pairs().head(10)

,CADASTRE_POLYGON_PID,GROUP_ID,ADDR_ID_A,SOURCE_ID_A,LABEL_A,STYLE_A,BLOCK_A,ADDR_ID_B,SOURCE_ID_B,LABEL_B,STYLE_B,BLOCK_B,RELATION,METHOD,P,BAND,EVIDENCE
0,cpl5a1700003dde,G2,A0000205,TXT40002665,"UNIT A101, 1-15 MERIDIAN AV",S1,S1.1,A0000206,TXT40002678,"TOWER A, UNIT 101, 1-15 MERIDIAN AV",S2,S2.1,SAME_DWELLING,same unit identifier,0.998,HIGH,same unit identifier explains 144 of 180/144 keys (+3.2); building ref A = A...
1,cpl5a1700003dde,G3,A0000208,TXT40002704,"UNIT A102, 1-15 MERIDIAN AV",S1,S1.1,A0000209,TXT40002717,"TOWER A, UNIT 102, 1-15 MERIDIAN AV",S2,S2.1,SAME_DWELLING,same unit identifier,0.998,HIGH,same unit identifier explains 144 of 180/144 keys (+3.2); building ref A = A...
2,cpl5a1700003dde,G4,A0000210,TXT40002730,"UNIT A103, 1-15 MERIDIAN AV",S1,S1.1,A0000211,TXT40002743,"TOWER A, UNIT 103, 1-15 MERIDIAN AV",S2,S2.1,SAME_DWELLING,same unit identifier,0.998,HIGH,same unit identifier explains 144 of 180/144 keys (+3.2); building ref A = A...
3,cpl5a1700003dde,G6,A0000215,TXT40002795,"UNIT A105, 1-15 MERIDIAN AV",S1,S1.1,A0000216,TXT40002808,"TOWER A, UNIT 105, 1-15 MERIDIAN AV",S2,S2.1,SAME_DWELLING,same unit identifier,0.998,HIGH,same unit identifier explains 144 of 180/144 keys (+3.2); building ref A = A...
4,cpl5a1700003dde,G7,A0000217,TXT40002821,"UNIT A106, 1-15 MERIDIAN AV",S1,S1.1,A0000218,TXT40002834,"TOWER A, UNIT 106, 1-15 MERIDIAN AV",S2,S2.1,SAME_DWELLING,same unit identifier,0.998,HIGH,same unit identifier explains 144 of 180/144 keys (+3.2); building ref A = A...
5,cpl5a1700003dde,G8,A0000220,TXT40002860,"UNIT A107, 1-15 MERIDIAN AV",S1,S1.1,A0000221,TXT40002873,"TOWER A, UNIT 107, 1-15 MERIDIAN AV",S2,S2.1,SAME_DWELLING,same unit identifier,0.998,HIGH,same unit identifier explains 144 of 180/144 keys (+3.2); building ref A = A...
6,cpl5a1700003dde,G9,A0000222,TXT40002886,"UNIT A108, 1-15 MERIDIAN AV",S1,S1.1,A0000223,TXT40002899,"TOWER A, UNIT 108, 1-15 MERIDIAN AV",S2,S2.1,SAME_DWELLING,same unit identifier,0.998,HIGH,same unit identifier explains 144 of 180/144 keys (+3.2); building ref A = A...
7,cpl5a1700003dde,G10,A0000224,TXT40002912,"UNIT A109, 1-15 MERIDIAN AV",S1,S1.1,A0000225,TXT40002925,"TOWER A, UNIT 109, 1-15 MERIDIAN AV",S2,S2.1,SAME_DWELLING,same unit identifier,0.998,HIGH,same unit identifier explains 144 of 180/144 keys (+3.2); building ref A = A...
8,cpl5a1700003dde,G11,A0000227,TXT40002951,"UNIT A110, 1-15 MERIDIAN AV",S1,S1.1,A0000228,TXT40002964,"TOWER A, UNIT 110, 1-15 MERIDIAN AV",S2,S2.1,SAME_DWELLING,same unit identifier,0.998,HIGH,same unit identifier explains 144 of 180/144 keys (+3.2); building ref A = A...
9,cpl5a1700003dde,G12,A0000230,TXT40002990,"UNIT A201, 1-15 MERIDIAN AV",S1,S1.1,A0000231,TXT40003003,"TOWER A, UNIT 201, 1-15 MERIDIAN AV",S2,S2.1,SAME_DWELLING,same unit identifier,0.998,HIGH,same unit identifier explains 144 of 180/144 keys (+3.2); building ref A = A...


In [17]:
rc = s.review_candidates()
print(rc.RELATION.value_counts().to_dict())
rc.head(6)

{'AMBIGUOUS_BUILDING': 172, 'REVIEW_RENUMBERING': 54}


,CADASTRE_POLYGON_PID,GROUP_ID,ADDR_ID_A,SOURCE_ID_A,LABEL_A,STYLE_A,BLOCK_A,ADDR_ID_B,SOURCE_ID_B,LABEL_B,STYLE_B,BLOCK_B,RELATION,METHOD,P,BAND,EVIDENCE
0,cpl5a1700007bbc,,A0001110,TXT40014430,"UNIT 101, 120 CANAL RD",S1,S1.1a,A0001111,TXT40014443,"UNIT 1, 120 CANAL RD",S1,S1.1b,REVIEW_RENUMBERING,unit number = position in the other scheme (e.g. strata lot order),0.6246,REVIEW,same style on two contexts (-3.0); unit number = position in the other schem...
1,cpl5a1700007bbc,,A0001112,TXT40014456,"UNIT 102, 120 CANAL RD",S1,S1.1a,A0001113,TXT40014469,"UNIT 2, 120 CANAL RD",S1,S1.1b,REVIEW_RENUMBERING,unit number = position in the other scheme (e.g. strata lot order),0.6246,REVIEW,same style on two contexts (-3.0); unit number = position in the other schem...
2,cpl5a1700007bbc,,A0001114,TXT40014482,"UNIT 103, 120 CANAL RD",S1,S1.1a,A0001115,TXT40014495,"UNIT 3, 120 CANAL RD",S1,S1.1b,REVIEW_RENUMBERING,unit number = position in the other scheme (e.g. strata lot order),0.6246,REVIEW,same style on two contexts (-3.0); unit number = position in the other schem...
3,cpl5a1700007bbc,,A0001116,TXT40014508,"UNIT 104, 120 CANAL RD",S1,S1.1a,A0001117,TXT40014521,"UNIT 4, 120 CANAL RD",S1,S1.1b,REVIEW_RENUMBERING,unit number = position in the other scheme (e.g. strata lot order),0.6246,REVIEW,same style on two contexts (-3.0); unit number = position in the other schem...
4,cpl5a1700007bbc,,A0001118,TXT40014534,"UNIT 105, 120 CANAL RD",S1,S1.1a,A0001119,TXT40014547,"UNIT 5, 120 CANAL RD",S1,S1.1b,REVIEW_RENUMBERING,unit number = position in the other scheme (e.g. strata lot order),0.6246,REVIEW,same style on two contexts (-3.0); unit number = position in the other schem...
5,cpl5a1700007bbc,,A0001120,TXT40014560,"UNIT 106, 120 CANAL RD",S1,S1.1a,A0001121,TXT40014573,"UNIT 6, 120 CANAL RD",S1,S1.1b,REVIEW_RENUMBERING,unit number = position in the other scheme (e.g. strata lot order),0.6246,REVIEW,same style on two contexts (-3.0); unit number = position in the other schem...


In [18]:
annotated = s.addresses()
annotated[annotated.GROUP_ID != ""].head(10)

,ADDR_ID,SOURCE_ID,CADASTRE_POLYGON_PID,LABEL,KIND,STYLE_ID,STYLE_TEMPLATE,BLOCK_ID,NUMBERING_MODEL,STRUCTURE_ID,LEVEL,UNIT,PARENT_ROLE,GROUP_ID,IS_PREFERRED,DUPLICATE_OF,GROUP_P_MIN,GROUP_BAND
0,A0000943,TXT40012259,cpl5a1700003dde,"TOWER C, UNIT 503, 1-15 MERIDIAN AV",SUB,S2,"TOWER {B}, UNIT {L}{UU}, {N}-{N} {STREET}",S2.3,encoded,B3,L5,3,,G287,False,A0000942,0.9980,HIGH
1,A0000296,TXT40003848,cpl5a1700003dde,"UNIT A409, 1-15 MERIDIAN AV",SUB,S1,"UNIT {B}{L}{UU}, {N}-{N} {STREET}",S1.1,encoded,B1,L4,9,,G37,True,,0.9980,HIGH
2,A0000431,TXT40005603,cpl5a1700003dde,"TOWER A, UNIT 1006, 1-15 MERIDIAN AV",SUB,S2,"TOWER {B}, UNIT {L}{UU}, {N}-{N} {STREET}",S2.1,encoded,B1,L10,6,,G87,False,A0000430,0.9980,HIGH
3,A0000925,TXT40012025,cpl5a1700003dde,"UNIT 401, 15 MERIDIAN AV",SUB,S3,"UNIT {L}{UU}, {N} {STREET}",S3.2,encoded,B3,L4,1,,G280,False,A0000923,0.9327,HIGH
4,A0000521,TXT40006773,cpl5a1700003dde,"UNIT 1404, 1 MERIDIAN AV",SUB,S3,"UNIT {L}{UU}, {N} {STREET}",S3.1,encoded,B1,L14,4,,G121,False,A0000519,0.9571,HIGH
5,A0000821,TXT40010673,cpl5a1700003dde,"UNIT B1002, 1-15 MERIDIAN AV",SUB,S1,"UNIT {B}{L}{UU}, {N}-{N} {STREET}",S1.2,encoded,B2,L10,2,,G240,True,,0.9980,HIGH
6,A0000559,TXT40007267,cpl5a1700003dde,"UNIT A1509, 1-15 MERIDIAN AV",SUB,S1,"UNIT {B}{L}{UU}, {N}-{N} {STREET}",S1.1,encoded,B1,L15,9,,G136,True,,0.9820,HIGH
7,A0000529,TXT40006877,cpl5a1700003dde,"TOWER A, UNIT 1407, 1-15 MERIDIAN AV",SUB,S2,"TOWER {B}, UNIT {L}{UU}, {N}-{N} {STREET}",S2.1,encoded,B1,L14,7,,G124,False,A0000528,0.9571,HIGH
8,A0000665,TXT40008645,cpl5a1700003dde,"UNIT 203, 7 MERIDIAN AV",SUB,S3,"UNIT {L}{UU}, {N} {STREET}",S3.3,encoded,B2,L2,3,,G178,False,A0000663,0.9478,HIGH
9,A0000484,TXT40006292,cpl5a1700003dde,"UNIT 1208, 1 MERIDIAN AV",SUB,S3,"UNIT {L}{UU}, {N} {STREET}",S3.1,encoded,B1,L12,8,,G107,False,A0000482,0.9571,HIGH


## 8 · Explain one address

In [19]:
aid = annotated[(annotated.KIND == "SUB") & (annotated.GROUP_ID != "")].ADDR_ID.iloc[0]
e = s.explain(aid)
e["PAIRS"]

ADDR_ID          A0000943
SOURCE_ID        TXT40012259
PARCEL           cpl5a1700003dde
LABEL            TOWER C, UNIT 503, 1-15 MERIDIAN AV
KIND             sub-address
STYLE            S2  TOWER {B}, UNIT {L}{UU}, {N}-{N} {STREET}
BLOCK            S2.3  1-15 MERIDIAN AV | TOWER C
NUMBERING_MODEL  unit number encodes level (1103 = level 11, unit 3); levels L1-L8, ~5 per level
KEYS             {'raw': '503', 'lu': (5.0, '3'), 'lr': (5.0, 2), 'seq': 20}
STRUCTURE        B3
PARENT_ROLE      
GROUP            G287 (HIGH, preferred A0000942)


,OTHER,OTHER_LABEL,RELATION,P,METHOD
0,A0000942,"UNIT C503, 1-15 MERIDIAN AV",SAME_DWELLING,0.998,same unit identifier


In [20]:
e["BLOCK_LINKS"]

,LINK,TRANSFORM,P,STATUS,EVIDENCE
0,S1.3~S2.3,same unit identifier,0.998,KEPT,same unit identifier explains 35 of 48/35 keys (+3.2); building ref C = C (+...
1,S2.3~S3.2,same unit identifier,0.933,KEPT,same unit identifier explains 20 of 35/28 keys (+1.8); range contains number...
2,S2.3~S3.3,same unit identifier,0.232,REJECTED,same unit identifier explains 27 of 35/61 keys (+0.5); smaller block spans o...
3,S2.3~S3.1,level + unit-on-level with level offset +1,0.078,REJECTED,level + unit-on-level with level offset +1 explains 24 of 35/102 keys (-0.8)...
4,S1.2~S2.3,same unit identifier,0.020,REJECTED,same unit identifier explains 35 of 96/35 keys (+1.6); smaller block spans o...
5,S1.1~S2.3,same unit identifier,0.010,REJECTED,same unit identifier explains 35 of 180/35 keys (+0.9); smaller block spans ...
6,S2.2~S2.3,same unit identifier,0.001,REJECTED,same style on two contexts (-3.0); same unit identifier explains 30 of 79/35...
7,S2.1~S2.3,level + unit-on-level with level offset -1,0.000,REJECTED,same style on two contexts (-3.0); level + unit-on-level with level offset -...


## 9 · Evaluate (synthetic data only)

Scored on the pairs implied by the duplicate groups. Two scenarios are *meant* to have zero grouped recall:
- `strata_lot_numbering`: needs strata-lot IDs to confirm;
- `twin_towers_unresolved`: nothing separates the towers.

Their true pairs should all appear in the review list.

In [21]:
ev = s.evaluate(truth)
ev["by_scenario"]

precision 1.0000  recall 0.8746  (TP 990, FP 0, FN 142); review candidates 226 (140 true); recall if reviewed 0.9982


,SCENARIO,PRECISION,RECALL,F1,TP,FP,FN,REVIEW_PAIRS,REVIEW_TRUE
0,background,1.0,1.0,1.0,24,0,0,0,0
1,corner_frontage,1.0,1.0,1.0,60,0,0,0,0
2,level_convention,1.0,1.0,1.0,36,0,0,0,0
3,mixed_use,1.0,1.0,1.0,58,0,0,0,0
4,multi_tower,1.0,1.0,1.0,605,0,0,0,0
5,single_tower,1.0,1.0,1.0,94,0,0,0,0
6,strata_lot_numbering,1.0,0.0,0.0,0,0,54,54,54
7,townhouse_mix,1.0,1.0,1.0,17,0,0,0,0
8,twin_towers,1.0,1.0,1.0,96,0,0,0,0
9,twin_towers_unresolved,1.0,0.0,0.0,0,0,88,172,86


## 10 · Tune and re-run

Every threshold and evidence weight lives in `Config`. `config.with_(...)` returns a modified copy. Re-running a subset of parcels replaces just those results.

In [22]:
print({k: v for k, v in s.config.to_dict().items() if k in ("link_threshold", "band_low", "band_medium", "band_high", "prior_same_style", "ambiguity_margin")})
strict = s.config.with_(band_low=0.85)                   # keep only HIGH-confidence pairs
s.run(parcels=[pid], config=strict)
print(s.parcel_summary().query("CADASTRE_POLYGON_PID == @pid")[["N_DUPLICATE_GROUPS", "N_REDUNDANT_ADDRESSES"]])
s.run(config=Config())                                   # back to defaults for everything

{'prior_same_style': -3.0, 'link_threshold': 0.5, 'ambiguity_margin': 1.5, 'band_high': 0.85, 'band_medium': 0.6, 'band_low': 0.4}


run: 1 parcels analysed in 0.12s, 299 duplicate groups
   N_DUPLICATE_GROUPS  N_REDUNDANT_ADDRESSES
0                 299                    449


run: 28 parcels analysed in 0.30s, 655 duplicate groups


AssociatorSession(2100 rows, 28 parcels analysed, 655 duplicate groups, 226 review pairs)

In [23]:
s

AssociatorSession(2100 rows, 28 parcels analysed, 655 duplicate groups, 226 review pairs)

## 11 · Hand the results on

`to_frames()` returns every output as a DataFrame. Write them wherever your workflow needs, e.g. `frames["duplicate_groups"].to_parquet(...)` or `spark.createDataFrame(...)`.

In [24]:
frames = s.to_frames()
{name: f.shape for name, f in frames.items()}

{'issues': (1, 5),
 'parcel_summary': (28, 12),
 'styles': (53, 8),
 'blocks': (67, 14),
 'mappings': (78, 15),
 'structures': (34, 9),
 'addresses': (2079, 18),
 'duplicate_pairs': (976, 17),
 'duplicate_groups': (655, 13),
 'review_candidates': (226, 17)}